In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import csv

In [ ]:
root_dir = '/content/drive/MyDrive/analysis_data/olist_analysis'

In [ ]:
os.chdir (root_dir)

In [66]:
customers = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
category_translation = pd.read_csv(
    'product_category_name_translation.csv'
)

In [ ]:
#dataframes sizes / размеры датафреймов
tables = {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'order_payments': order_payments,
    'order_reviews': order_reviews,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

for name, df in tables.items():
    print(f'{name}: {df.shape}')

customers: (57706, 5)
orders: (99441, 8)
order_items: (112650, 7)
order_payments: (94280, 5)
order_reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (73, 2)


In [ ]:
# dtypes / типы данных
for name, df in tables.items():
    print(f'\n{name}')
    print(df.dtypes)


customers
customer_id                  object
customer_unique_id           object
customer_zip_code_prefix    float64
customer_city                object
customer_state               object
dtype: object

orders
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

order_items
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

order_payments
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments    float64
payment_value           float64
dtype: object

order_reviews
revi

In [ ]:
#missing values / пропуски
for name, df in tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f'\n{name}')
    print(missing)


customers
customer_zip_code_prefix    1
customer_city               1
customer_state              1
dtype: int64

orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

order_items
Series([], dtype: int64)

order_payments
payment_installments    1
payment_value           1
dtype: int64

order_reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

sellers
Series([], dtype: int64)

category_translation
Series([], dtype: int64)


In [ ]:
# doublicates / дубликаты
for name, df in tables.items():
    print(
        f'{name}: {df.duplicated().sum()}'
    )

customers: 0
orders: 0
order_items: 0
order_payments: 0
order_reviews: 0
products: 0
sellers: 0
category_translation: 0


In [ ]:
# dates transformation / преобразование дат
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [ ]:
# adding some features / добавим полезыне признаки
orders['purchase_month'] = (
    orders['order_purchase_timestamp']
    .dt.to_period('M')
    .astype(str)
)

orders['purchase_year'] = (
    orders['order_purchase_timestamp'].dt.year
)

In [ ]:
# combining the Portuguese category names with the English one / Объединим португальские названия категорий с английскими
products = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

In [ ]:
# main table for EDA / основная таблица для анализа
df = orders.merge(
    customers,
    on='customer_id',
    how='left'
)
df = df.merge(
    order_items,
    on='order_id',
    how='left'
)
df = df.merge(
    products,
    on='product_id',
    how='left'
)

In [ ]:
df.shape

(113425, 29)

In [64]:
# dalivery features / признаки доставки

# delivery days / время доставки
df['delivery_days'] = (
    df['order_delivered_customer_date']
    - df['order_purchase_timestamp']
).dt.total_seconds() / 86400

# delay days / задержка в днях
df['delivery_delay_days'] = (
    df['order_delivered_customer_date']
    - df['order_estimated_delivery_date']
).dt.total_seconds() / 86400

# delay bool / задержка
df['is_late'] = df['delivery_delay_days'] > 0

In [65]:
# exporting csv / экспорт csv
df.to_csv('df', index='False', encoding='utf-8')